
# Enviar un CSV por WiFi a los servos (ESP32 + PCA9685)

1. Conecta el PC a la red WiFi del ESP32 (`ERIS_SERVO`, IP `192.168.4.1`, puerto `8888`) con el sketch `eris_servo_wifi.ino`.
2. Mapeo por **posicion de columna**: la primera columna de datos va al canal 0 del PCA9685, la segunda al canal 1, etc. (max. 16). Los nombres de las columnas no importan.
3. Si el CSV trae una columna de tiempo (`t`, `t_s`, `time`, `time_s`, `t_ms`) en cualquier posicion, **no** se envia a ningun canal: se usa solo para el ritmo de reproduccion. Si no hay columna de tiempo, se usa un paso fijo `RATE_HZ`.
4. Los valores se envian tal cual como angulo de servo (0-180, el firmware recorta fuera de rango). Si tu CSV tiene otra convencion (p. ej. angulos MuJoCo negativos), ajusta `OFFSET`/`SIGN` en la celda de configuracion.


In [28]:

import csv, socket, threading, time
from pathlib import Path

HOST, PORT = "192.168.4.1", 8888
CSV_PATH = "leg_actuator_angles_trajectory_adapted.csv"   # <- cambia al CSV a enviar

RATE_HZ = 50          # solo si el CSV no tiene columna de tiempo
TIME_SCALE = 1.0      # >1 = mas lento, <1 = mas rapido
REPEAT = 1            # numero de repeticiones del CSV
MAX_CHANNELS = 16

# Correccion opcional por columna (lista de MAX_CHANNELS o un solo numero para todos):
# angulo_enviado = SIGN * valor_csv + OFFSET
SIGN = 1
OFFSET = 0

TIME_COLUMNS = ("t", "t_s", "time", "time_s", "t_ms")


In [29]:

class ErisWifi:
    """Cliente TCP del protocolo de texto del ESP32. Lee las respuestas en un hilo."""
    def __init__(self, host, port, timeout=5):
        self.sock = socket.create_connection((host, port), timeout=timeout)
        self.sock.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
        self.sock.settimeout(None)
        self.replies, self.errors = 0, []
        self._alive = True
        threading.Thread(target=self._reader, daemon=True).start()

    def _reader(self):
        buf = b""
        while self._alive:
            try:
                data = self.sock.recv(4096)
            except OSError:
                break
            if not data:
                break
            buf += data
            while b"\n" in buf:
                line, buf = buf.split(b"\n", 1)
                line = line.decode(errors="replace").strip()
                self.replies += 1
                if line.startswith("Error"):
                    self.errors.append(line)

    def send(self, text):
        self.sock.sendall(text.encode())

    def close(self):
        self._alive = False
        try: self.sock.close()
        except OSError: pass

def per_channel(v, n):
    return list(v) if isinstance(v, (list, tuple)) else [v] * n

def load_csv(path):
    with open(path, newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        rows = [r for r in reader if r and any(c.strip() for c in r)]
    time_idx = next((i for i, h in enumerate(header) if h.strip().lower() in TIME_COLUMNS), None)
    data_idx = [i for i in range(len(header)) if i != time_idx][:MAX_CHANNELS]
    times = None
    if time_idx is not None:
        times = [float(r[time_idx]) for r in rows]
        if header[time_idx].strip().lower() == "t_ms":
            times = [t / 1000 for t in times]
    values = [[float(r[i]) for i in data_idx] for r in rows]
    names = [header[i] for i in data_idx]
    return names, times, values

names, times, values = load_csv(CSV_PATH)
n_ch = len(names)
print(f"{len(values)} filas, {n_ch} canales, tiempo: {'columna' if times else f'{RATE_HZ} Hz fijo'}")
for ch, n in enumerate(names):
    print(f"  canal {ch:2d} <- columna '{n}'")
out = [n for n, col in zip(names, zip(*values)) if min(col) < 0 or max(col) > 180]
if out:
    print(f"\nOJO: {len(out)} columnas salen de 0-180 y el firmware las recortaria: {out}")
    print("Convierte a angulo de servo (SIGN/OFFSET) antes de reproducir en el robot.")


12065 filas, 12 canales, tiempo: columna
  canal  0 <- columna 'fr_hip_joint'
  canal  1 <- columna 'fr_knee_joint'
  canal  2 <- columna 'fr_ankle_joint'
  canal  3 <- columna 'fl_hip_joint'
  canal  4 <- columna 'fl_knee_joint'
  canal  5 <- columna 'fl_ankle_joint'
  canal  6 <- columna 'br_hip_joint'
  canal  7 <- columna 'br_knee_joint'
  canal  8 <- columna 'br_ankle_joint'
  canal  9 <- columna 'bl_hip_joint'
  canal 10 <- columna 'bl_knee_joint'
  canal 11 <- columna 'bl_ankle_joint'

OJO: 6 columnas salen de 0-180 y el firmware las recortaria: ['br_hip_joint', 'br_knee_joint', 'br_ankle_joint', 'bl_hip_joint', 'bl_knee_joint', 'bl_ankle_joint']
Convierte a angulo de servo (SIGN/OFFSET) antes de reproducir en el robot.


In [30]:

SIGNS, OFFSETS = per_channel(SIGN, n_ch), per_channel(OFFSET, n_ch)

def row_to_cmd(row, cmd="X"):
    # Un comando por canal, en un solo envio TCP (no hace falta esperar cada respuesta).
    return "".join(f"{cmd} {ch} {SIGNS[ch] * v + OFFSETS[ch]:.2f}\n" for ch, v in enumerate(row))

def play(dev, times, values, repeat=1, scale=1.0, rate_hz=50, go_to_start=True):
    if go_to_start:   # rampa suave (Y) a la primera fila antes de reproducir
        dev.send(row_to_cmd(values[0], "Y"))
        print("Yendo a la posicion inicial...")
        time.sleep(3)
    t0 = time.monotonic()
    for rep in range(repeat):
        base = t0 + (rep * (times[-1] + (times[1] - times[0]) if times else len(values) / rate_hz)) * scale
        for k, row in enumerate(values):
            target = base + (times[k] * scale if times else k / rate_hz * scale)
            wait = target - time.monotonic()
            if wait > 0:
                time.sleep(wait)
            dev.send(row_to_cmd(row))
        print(f"Repeticion {rep + 1}/{repeat} enviada")

dev = ErisWifi(HOST, PORT)
dev.send("INFO\n"); time.sleep(0.3)
print("Conectado. Respuestas recibidas:", dev.replies)


Conectado. Respuestas recibidas: 0


In [31]:

try:
    play(dev, times, values, repeat=REPEAT, scale=TIME_SCALE, rate_hz=RATE_HZ)
except KeyboardInterrupt:
    print("Detenido por el usuario")
time.sleep(0.3)
print(f"Respuestas: {dev.replies}, errores: {len(dev.errors)}")
print(dev.errors[:5])


Yendo a la posicion inicial...
Detenido por el usuario
Respuestas: 23780, errores: 0
[]


In [32]:

dev.close()
